In [ ]:
from openai import OpenAI
from pydantic import BaseModel, Field
from google.colab import userdata, drive
from typing import List, Union, Set
import os
import json
from tqdm import tqdm
import pickle

drive.mount('/content/drive')
grok  = OpenAI(api_key=userdata.get('grok'), base_url="https://api.x.ai/v1")

In [ ]:
outline_prompt = """You will be given parts of a research paper in JSON format and a list of attempted methods in JSON format.

Instructions:

- You are an expert researcher.
"""

writing_prompt  = """You will be given JSON formatted text.

Instructions:
- You will write the complete methodology section of a research paper in paragraph format.
- Use a formal and direct tone for the paper.
- Explain ALL steps logically with well-defined connections between ideas and sections.
- Elaborate heavily on the the 'experimental_plan'. Include extreme detail and thoroughness.
- Include specific examples with detailed explanations for further elaboration.
- Write all mathematical expressions in LaTeX.
- Be EXTREMELY verbose and thorough.
- Do NOT use excessive subsections. Instead, connect certain concepts within a section in a smooth way.
"""

In [ ]:
class Section(BaseModel):
    name: str
    content: str

class Outline(BaseModel):
    proposed_method: str = Field(..., description="""Using the given information, first provide inspiration behind a new proposed method to address the main research problem.
                                                    You should also motivate why the proposed method would work better than existing works. Then, explain how the proposed
                                                    approach works, and describe all the essential steps. Do NOT repeat proposed methods that are already in the 'attempted_methods.'""")
    experimental_plan: str = Field(..., description="""Break down EVERY single step in 'proposed_method'. Every step MUST be executable.
                                                    Cover ALL essential details such as the datasets, models, metrics to be used, etc.""")
class Contributions(BaseModel):
    contributions: List[Section] = Field(..., description="The contributions section will include ALL of the following sections: Methods, Experiments.")

In [ ]:
def call_grok(messages, model_name, response_format, temperature=0.7, max_tokens=4096):
    if response_format is None:
        completions = grok.beta.chat.completions.parse(
            model=model_name,
            messages=messages,
            max_completion_tokens=max_tokens,
            temperature=temperature
        )
    else:
        completions = grok.beta.chat.completions.parse(
            model=model_name,
            messages=messages,
            max_completion_tokens=max_tokens,
            response_format=response_format,
            temperature=temperature
        )
    return completions.choices[0].message.content

def outline(redacted_paper, attempted_methods):
    messages = [
        {"role":"system", "content": outline_prompt},
        {"role": "user", "content": redacted_paper},
        {"role": "assistant", "content": attempted_methods}
    ]
    return call_grok(messages, "grok-2-1212", Outline, 0.8)

def write_contributions(outline_paper):
    messages = [
        {"role":"system", "content": writing_prompt},
        {"role": "user", "content": outline_paper}
    ]
    return call_grok(messages, "grok-2-1212", Contributions, 0.8)

In [ ]:
redacted_papers = pickle.load(open('/content/drive/MyDrive/ResearchPapers/manual_redaction_interface/redacted_papers.pkl', 'rb'))
tries = 1

base_dir = '/content/drive/MyDrive/ResearchPapers/grok_official_predictions'
predictions = []
os.makedirs(base_dir, exist_ok=True)

def run(starting_idx):
    idx_list = list(range(starting_idx, 101))

    while idx_list:
        idx = idx_list[0]  # Always try the first remaining index
        try:
            print(f"Processing paper {idx}...")
            custom_id, redacted_paper = redacted_papers[idx]
            paper_dir = os.path.join(base_dir, custom_id)
            os.makedirs(paper_dir, exist_ok=True)

            attempted_methods = []
            for i in range(tries):
                outline_paper = outline(redacted_paper, json.dumps({"attempted_methods": attempted_methods}))
                print(f'Done with outline paper for paper {idx} for try {i+1}.')
                with open(os.path.join(paper_dir, f'outline_{i+1}.txt'), "w") as f:
                    f.write(outline_paper)

                attempted_methods.append(json.loads(outline_paper)['proposed_method'])

                contributions = write_contributions(outline_paper)
                print(f'Done with contributions for paper {idx} for try {i+1}.')
                predictions.append(contributions)
                with open(os.path.join(paper_dir, f'contributions_{i+1}.txt'), "w") as f:
                    f.write(contributions)
            idx_list.pop(0)

        except Exception as e:
            print(f"Error at index {idx}: {e}")
            print("Retrying from the same index...")
run(0)

In [ ]:
with open(os.path.join(base_dir, 'predictions.pkl'), 'wb') as f:
    pickle.dump(predictions, f)